# Corss validation of China

## I. Validation using time series

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn.inspection import partial_dependence
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.datasets import make_moons
from sklearn.preprocessing import MinMaxScaler
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import RocCurveDisplay

In [ ]:
re_data = pd.read_csv(r'D:\Data\re_data.csv')
production_mode = pd.read_csv(r'D:\Data\plantmode_ml.csv')
re_data = pd.merge(re_data, production_mode, on='name_prod', how='left')
re_data = re_data.drop(['name_book', 'products'], axis=1)
re_data = re_data.dropna()

In [ ]:
X_plant = re_data.drop(['Iron_Prod', 'Steel_Prod', 'name_prod', 'IDCode', 'plant_id','Date'], axis=1)
y_needed_plant = np.log1p(re_data['Steel_Prod']) 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, make_scorer, r2_score
import xgboost as xgb


X_train, X_test, y_train, y_test = train_test_split(X_plant, y_needed_plant, test_size=0.2, random_state=1)


pollutants = ['Plant_CO_MEAN', 'Plant_NO2_MEAN', 'Plant_PM2_5_MEAN', 'Plant_PM10_MEAN', 
              'Plant_SO2_MEAN', 'Plant_LSTA_MEAN', 'Plant_LSTT_MEAN', 'Plant_NTL_MEAN', 'Plant_O3_MEAN']

other_features = [col for col in X_plant.columns if col not in pollutants + ['Longitude', 'Latitude']]

preprocessor = ColumnTransformer(
    transformers=[
        ('pollutants', StandardScaler(), pollutants),  
        ('spatial', RobustScaler(), ['Longitude', 'Latitude']),  
        ('others', StandardScaler(), other_features) 
    ])

xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgb', xgb.XGBRegressor(
        colsample_bytree=0.6,  
        learning_rate=0.1,
        max_depth=6,
        n_estimators=300,
        subsample=0.8,
        alpha=0.2, 
        reg_lambda=0.5, 
        random_state=5
    ))
])


def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def calculate_wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

kf = KFold(n_splits=5, shuffle=True, random_state=1)

mse_scores = cross_val_score(xgb_model, X_plant, y_needed_plant, cv=kf, scoring=make_scorer(mean_squared_error))
rmse_scores = cross_val_score(xgb_model, X_plant, y_needed_plant, cv=kf, scoring=make_scorer(calculate_rmse))
mape_scores = cross_val_score(xgb_model, X_plant, y_needed_plant, cv=kf, scoring=make_scorer(calculate_mape))
wmape_scores = cross_val_score(xgb_model, X_plant, y_needed_plant, cv=kf, scoring=make_scorer(calculate_wmape))
r2_scores = cross_val_score(xgb_model, X_plant, y_needed_plant, cv=kf, scoring='r2')

results_xgb = {
    "Model": "XGBoost",
    "MSE": np.mean(mse_scores),
    "RMSE": np.mean(rmse_scores),
    "MAPE": np.mean(mape_scores),
    "WMAPE": np.mean(wmape_scores),
    "R2 Score": np.mean(r2_scores)
}

results_df_xgb = pd.DataFrame([results_xgb])
print(results_df_xgb)

In [ ]:
import folium
from branca.colormap import LinearColormap

china_map = folium.Map(location=[35.8617, 104.1954], zoom_start=5,  tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', attr='Esri')

colormap = LinearColormap(
    colors=['blue', 'green', 'yellow', 'orange', 'red'],
    vmin=re_data['Steel_Prod'].min(),
    vmax=re_data['Steel_Prod'].max(),
    caption='Steel Production (tppa)'
)

for index, row in re_data.iterrows():
    lat = row['Latitude']
    lon = row['Longitude']
    steel_output = row['Steel_Prod']

    color = colormap(steel_output)

    radius = (steel_output ** 0.5) * 1  

    folium.CircleMarker(
        location=[lat, lon],
        radius=radius, 
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.2,  
        popup=(
            f"<b>Plant ID:</b> {row['plant_id']}<br>"
            f"<b>Steel Output:</b> {steel_output:.2f} tppa"
        )
    ).add_to(china_map)

colormap.add_to(china_map)

legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 250px;
    height: 150px;
    background-color: white;
    border: 2px solid grey;
    z-index: 1000;
    padding: 10px;
    font-size: 14px;">
    <b>Circle Size Legend:</b><br>
    <i style="background: #ccc; border-radius: 50%; width: 10px; height: 10px; display: inline-block;"></i> Small Production<br>
    <i style="background: #ccc; border-radius: 50%; width: 20px; height: 20px; display: inline-block;"></i> Medium Production<br>
    <i style="background: #ccc; border-radius: 50%; width: 30px; height: 30px; display: inline-block;"></i> Large Production<br>
</div>
"""
china_map.get_root().html.add_child(folium.Element(legend_html))

china_map


## I.A use 2019-2021 to train and test model, and 2022 as out-of-sample

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

train_data = re_data[re_data['Year'].isin([2019, 2020, 2021])]
validation_data = re_data[re_data['Year'] == 2022]

X_train = train_data.drop(['Steel_Prod', 'name_prod', 'Iron_Prod', 'IDCode', 'plant_id','Date'], axis=1)
y_train = np.log1p(train_data['Steel_Prod'])  

X_validation = validation_data.drop(['Steel_Prod', 'name_prod', 'Iron_Prod', 'IDCode', 'plant_id'], axis=1)
y_validation = validation_data['Steel_Prod']

pollutants = ['Plant_CO_MEAN', 'Plant_NO2_MEAN', 'Plant_PM2_5_MEAN', 'Plant_PM10_MEAN', 
              'Plant_SO2_MEAN', 'Plant_LSTA_MEAN', 'Plant_LSTT_MEAN', 'Plant_NTL_MEAN', 'Plant_O3_MEAN']

other_features = [col for col in X_train.columns if col not in pollutants + ['Longitude', 'Latitude']]

preprocessor = ColumnTransformer(
    transformers=[
        ('pollutants', StandardScaler(), pollutants), 
        ('spatial', RobustScaler(), ['Longitude', 'Latitude']),  
        ('others', StandardScaler(), other_features)  
    ])

xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgb', xgb.XGBRegressor(
        colsample_bytree=0.6,
        learning_rate=0.1,
        max_depth=6,
        n_estimators=300,
        subsample=0.8,
        alpha=0.2,
        reg_lambda=0.5,
        random_state=5
    ))
])

xgb_model.fit(X_train, y_train)

y_pred_validation = xgb_model.predict(X_validation)

y_pred_validation = np.expm1(y_pred_validation)

mse = mean_squared_error(y_validation, y_pred_validation)
rmse = np.sqrt(mse)
r2 = r2_score(y_validation, y_pred_validation)

kf = KFold(n_splits=5, shuffle=True, random_state=1)
cross_val_scores = cross_val_score(xgb_model, X_train, y_train, cv=kf, scoring='r2')
print(f"Average R2 score from cross-validation (2019-2021): {np.mean(cross_val_scores):.4f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(6, 6))

plt.scatter(y_validation, y_pred_validation, alpha=0.7, color='blue', label='Predicted vs. Actual')

plt.plot([y_validation.min(), y_validation.max()], [y_validation.min(), y_validation.max()], 'r--', lw=2, label='Perfect Fit Line')

ticks = [0.1, 1, 10, 100] 
plt.xscale('log')
plt.yscale('log')
plt.xticks(ticks, labels=[str(int(tick)) for tick in ticks])
plt.yticks(ticks, labels=[str(int(tick)) for tick in ticks])

plt.xlabel('Actual Steel Production (10,000 tones)')
plt.ylabel('Predicted Steel Production (10,000 tones)')
plt.title('Predicted vs Actual Steel Output in 2022')
plt.legend()

plt.savefig(r'D:\Figures\pred_actual_2022.pdf', format='pdf', dpi=800)

plt.show()

## Visualize the Error Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_residuals = y_train - y_train_pred
test_residuals = y_validation - y_test_pred

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.histplot(train_residuals, kde=True, color='blue', bins=30)
plt.title('Training Error Distribution')
plt.xlabel('Residuals (Errors)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.histplot(test_residuals, kde=True, color='red', bins=30)
plt.title('Test Error Distribution')
plt.xlabel('Residuals (Errors)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()


## II. Validation across regions: South validation


In [ ]:
import pandas as pd
import folium
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb

re_data = pd.read_csv(r'D:\Data\re_data.csv')
production_mode = pd.read_csv(r'D:\Data\plantmode_ml.csv')
re_data = pd.merge(re_data, production_mode, on='name_prod', how='left')
re_data = re_data.drop(['name_book', 'products','Date'], axis=1)
re_data = re_data.dropna()

In [ ]:
def assign_geographical_region(row):
    lat = row['Latitude']
    lon = row['Longitude']
    
    if lat >= 35:
        if lon >= 115:
            return 'Northeast'
        else:
            return 'Northwest'
    else:
        if lon >= 115:
            return 'Southeast'
        else:
            return 'Southwest'

re_data['Region'] = re_data.apply(assign_geographical_region, axis=1)


print(re_data['Region'].value_counts())


china_map = folium.Map(location=[35.8617, 104.1954], zoom_start=5, 
                       tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', 
                       attr='Esri')

region_colors = {
    'Northeast': 'blue',
    'Northwest': 'green',
    'Southeast': 'orange',
    'Southwest': 'red'
}

for index, row in re_data.iterrows():
    lat = row['Latitude']
    lon = row['Longitude']
    region = row['Region']
    color = region_colors[region]
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"Plant ID: {row['plant_id']}<br>Region: {region}"
    ).add_to(china_map)

china_map.save('D:/Figures/china_steel_plant_geographical_regions_map.html')
china_map

In [ ]:

training_regions = ['Northeast', 'Northwest']
validation_regions = ['Southwest', 'Southeast']

train_data = re_data[re_data['Region'].isin(training_regions)]
validation_data = re_data[re_data['Region'].isin(validation_regions)]

X_train = train_data.drop(['Steel_Prod', 'name_prod', 'Iron_Prod', 'IDCode', 'plant_id', 'Region'], axis=1)
y_train = np.log1p(train_data['Steel_Prod'])

X_validation = validation_data.drop(['Steel_Prod', 'name_prod', 'Iron_Prod', 'IDCode', 'plant_id', 'Region'], axis=1)
y_validation = validation_data['Steel_Prod']

pollutants = ['Plant_CO_MEAN', 'Plant_NO2_MEAN', 'Plant_PM2_5_MEAN', 'Plant_PM10_MEAN', 
              'Plant_SO2_MEAN', 'Plant_LSTA_MEAN', 'Plant_LSTT_MEAN', 'Plant_NTL_MEAN', 'Plant_O3_MEAN']

other_features = [col for col in X_train.columns if col not in pollutants + ['Longitude', 'Latitude']]

preprocessor = ColumnTransformer(
    transformers=[
        ('pollutants', StandardScaler(), pollutants),  
        ('spatial', RobustScaler(), ['Longitude', 'Latitude']),  
        ('others', StandardScaler(), other_features)  
    ])

xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgb', xgb.XGBRegressor(
        colsample_bytree=0.6,
        learning_rate=0.1,
        max_depth=6,
        n_estimators=300,
        subsample=0.8,
        alpha=0.2,
        reg_lambda=0.5,
        random_state=5
    ))
])


xgb_model.fit(X_train, y_train)

y_pred_validation = xgb_model.predict(X_validation)

y_pred_validation = np.expm1(y_pred_validation)

mse = mean_squared_error(y_validation, y_pred_validation)
rmse = np.sqrt(mse)
r2 = r2_score(y_validation, y_pred_validation)

kf = KFold(n_splits=10, shuffle=True, random_state=1)
cross_val_scores = cross_val_score(xgb_model, X_train, y_train, cv=kf, scoring='r2')
print(f"Average R2 score from cross-validation: {np.mean(cross_val_scores):.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict
import numpy as np

y_train_pred = cross_val_predict(xgb_model, X_train, y_train, cv=kf)

y_train_pred_original = np.expm1(y_train_pred)  
y_train_original = np.expm1(y_train) 

plt.figure(figsize=(8, 8))
plt.scatter(y_validation, y_pred_validation, alpha=0.7, color='blue', label='Predicted vs. Actual')
plt.plot([y_validation.min(), y_validation.max()], [y_validation.min(), y_validation.max()], 
         'r--', lw=2, label='Perfect Fit Line')
plt.xscale('log')  
plt.yscale('log') 
ticks = [0.1, 1, 10, 100] 
plt.xticks(ticks, labels=[str(int(tick)) for tick in ticks])
plt.yticks(ticks, labels=[str(int(tick)) for tick in ticks])

plt.xlabel('Actual Steel Production (10,000 tones)')
plt.ylabel('Predicted Steel Production (10,000 tones)')
plt.title('Predicted vs Actual Steel Production Steel Output in Hold-out Sample')
plt.legend()

plt.savefig(r'D:\Figures\pred_actual_south.pdf', format='pdf', dpi=800)
plt.show()


In [ ]:
validation_data['Predicted_Steel_Output'] = y_pred_validation
validation_data['Error_Rate'] = abs(validation_data['Steel_Prod'] - validation_data['Predicted_Steel_Output']) / validation_data['Steel_Prod'] * 100
validation_data.info()